In [ ]:
# --- Librerias ---
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import Workbook

In [ ]:
# --- Parametros y rutas ---
ruta_watershed = r"datos/SS4_WS.tif"
ruta_original = r"datos/SS4.tif"
carpeta_salida = r"resultados"

# FOV real del microscopio (micras): no impreso en la captura, estimado por
# extrapolacion a partir del diametro mediano de fibra frente al Caso 1 (169x132 um)
fov_ancho_um = 534.0
fov_alto_um = 400.0

factor_motas = 0.025       # descarta piezas < 2.5% del area mediana
factor_fragmento = 0.9     # pieza < 0.9*mediana del subconjunto -> fragmento de sobre-seg
umbral_solidez = 0.88      # une dos piezas si su union llena >= 88% de su casco convexo
factor_max_grupo = 1.2     # no unir si el grupo supera 1.2x la mediana (particulas uniformes; no agranda)
factor_min_grupo = 0.4     # descarta grupos finales < 0.4x la mediana (particulas inventadas)
umbral_solape = 0.65       # fusiona 2 circulos si dist. entre centros < 0.65*(r1+r2) (se solapan demasiado)
factor_max_circulo = 1.3   # el circulo fusionado no supera 1.3x el radio mediano (no crea gigantes)

os.makedirs(carpeta_salida, exist_ok=True)

In [ ]:
# --- Cargar la mascara sobre-segmentada (watershed de ImageJ) ---
ws = cv2.imread(ruta_watershed, cv2.IMREAD_GRAYSCALE)
_, ws = cv2.threshold(ws, 127, 255, cv2.THRESH_BINARY)
# auto-inversion: las particulas deben ser el foreground (mas componentes). Si vienen
# como fondo (binarizado al reves en ImageJ) se corrige solo.
if cv2.connectedComponents(ws)[0] < cv2.connectedComponents(255 - ws)[0]:
    ws = 255 - ws
    print("Binario invertido -> corregido (particulas como foreground)")
alto_px, ancho_px = ws.shape

micras_por_pixel_x = fov_ancho_um / ancho_px
micras_por_pixel_y = fov_alto_um / alto_px
micras_por_pixel = (micras_por_pixel_x + micras_por_pixel_y) / 2

def contorno_exterior(mascara):
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return max(contornos, key=cv2.contourArea) if contornos else None

In [ ]:
# --- Piezas (sin motas) y clasificacion interior / borde ---
# Las piezas que tocan el marco son de borde: se tratan APARTE para que sus
# fragmentos no contaminen las interiores.
numero, etiquetas_piezas = cv2.connectedComponents(ws)
area = {}
contorno = {}
for pieza in range(1, numero):
    c = contorno_exterior(np.uint8(etiquetas_piezas == pieza) * 255)
    if c is None:
        continue
    area[pieza] = int((etiquetas_piezas == pieza).sum())
    contorno[pieza] = c

area_mediana = np.median(list(area.values()))
piezas = [p for p in area if area[p] >= factor_motas * area_mediana]

bordes_lab = set(np.unique(np.concatenate([etiquetas_piezas[0, :], etiquetas_piezas[-1, :],
                                           etiquetas_piezas[:, 0], etiquetas_piezas[:, -1]])))
piezas_interior = [p for p in piezas if p not in bordes_lab]
piezas_borde = [p for p in piezas if p in bordes_lab]
print("Piezas:", len(piezas), "| interior:", len(piezas_interior), "| borde:", len(piezas_borde))

In [ ]:
# --- Union por convexidad (interiores y bordes por separado) ---
piezas_set = set(piezas)
valido_mask = np.isin(etiquetas_piezas, piezas)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
vecinos = {}
for p in piezas:
    cerca = np.unique(etiquetas_piezas[(cv2.dilate(np.uint8(etiquetas_piezas == p), kernel) > 0) & valido_mask])
    vecinos[p] = [int(v) for v in cerca if v != p and v in piezas_set]

def solidez_union(a, b):
    casco = cv2.contourArea(cv2.convexHull(np.vstack([contorno[a], contorno[b]])))
    return (area[a] + area[b]) / casco if casco > 0 else 0.0

def unir(subconjunto):
    sub = set(subconjunto)
    if not subconjunto:
        return {}
    mediana = np.median([area[p] for p in subconjunto])
    tope = factor_max_grupo * mediana
    padre = {p: p for p in subconjunto}
    area_grupo = {p: area[p] for p in subconjunto}
    def raiz(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x
    for fragmento in sorted([p for p in subconjunto if area[p] < factor_fragmento * mediana], key=lambda p: area[p]):
        candidatos = []
        for v in vecinos[fragmento]:
            if v not in sub:
                continue
            rf, rv = raiz(fragmento), raiz(v)
            if rf == rv or area_grupo[rf] + area_grupo[rv] > tope:   # tope: no fundir dos particulas
                continue
            s = solidez_union(fragmento, v)
            if s > umbral_solidez:
                candidatos.append((s, v))
        if candidatos:
            rf, rv = raiz(fragmento), raiz(max(candidatos)[1])
            padre[rf] = rv
            area_grupo[rv] += area_grupo[rf]
    grupos = {}
    for p in subconjunto:
        grupos.setdefault(raiz(p), []).append(p)
    # descarta grupos diminutos (particulas inventadas)
    return {k: m for k, m in grupos.items() if sum(area[p] for p in m) >= factor_min_grupo * mediana}

grupos_interior = unir(piezas_interior)
grupos_borde = unir(piezas_borde)
print("Particulas interiores:", len(grupos_interior), "| de borde:", len(grupos_borde))

In [ ]:
# --- Interiores como circulos de area REAL + fusion de solapes; borde = casco convexo ---
mascara_exterior = np.zeros((alto_px, ancho_px), dtype=np.uint8)
for miembros in grupos_borde.values():
    cv2.drawContours(mascara_exterior, [cv2.convexHull(np.vstack([contorno[p] for p in miembros]))], -1, 255, -1)

# un circulo (centro, radio de area equivalente) por grupo interior
circulos = []
for miembros in grupos_interior.values():
    casco = cv2.convexHull(np.vstack([contorno[p] for p in miembros]))
    area_px = sum(area[p] for p in miembros)   # area REAL (pixeles), no la del casco (que infla)
    perimetro = cv2.arcLength(casco, True)
    circ = 4 * np.pi * cv2.contourArea(casco) / (perimetro ** 2) if perimetro > 0 else 0.0
    M = cv2.moments(casco)
    cx = M["m10"] / M["m00"] if M["m00"] else 0.0
    cy = M["m01"] / M["m00"] if M["m00"] else 0.0
    circulos.append([cx, cy, float(np.sqrt(area_px / np.pi)), circ])

# Fusion de circulos que se solapan DEMASIADO (una particula partida en dos):
# se sustituyen por un circulo que engloba a ambos, sin superar factor_max_circulo * radio mediano.
mediana_radio = float(np.median([c[2] for c in circulos])) if circulos else 0.0
tope_radio = factor_max_circulo * mediana_radio

def envolvente(a, b):
    x1, y1, r1 = a[:3]; x2, y2, r2 = b[:3]
    d = float(np.hypot(x2 - x1, y2 - y1))
    if d + min(r1, r2) <= max(r1, r2):          # uno ya contiene al otro
        return list(a[:3]) if r1 >= r2 else list(b[:3])
    R = (d + r1 + r2) / 2                        # circulo minimo que engloba ambos
    t = (R - r1) / d if d > 0 else 0.0
    return [x1 + (x2 - x1) * t, y1 + (y2 - y1) * t, R]

padre = list(range(len(circulos)))
def raiz(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
for i in range(len(circulos)):
    xi, yi, ri = circulos[i][:3]
    for j in range(i + 1, len(circulos)):
        xj, yj, rj = circulos[j][:3]
        if np.hypot(xj - xi, yj - yi) < umbral_solape * (ri + rj):
            padre[raiz(i)] = raiz(j)

componentes = {}
for i in range(len(circulos)):
    componentes.setdefault(raiz(i), []).append(circulos[i])

circulos_final = []
for grupo in componentes.values():
    c = grupo[0]
    for otro in grupo[1:]:
        e = envolvente(c, otro)
        if e[2] <= tope_radio:                  # no crear circunferencias gigantes
            c = [e[0], e[1], e[2], max(c[3], otro[3])]
        else:
            circulos_final.append(otro)
    circulos_final.append(c)

# rasterizar interiores (circulos ya fusionados) y guardar propiedades
mascara_interior = np.zeros((alto_px, ancho_px), dtype=np.uint8)
particulas = []
for cx, cy, r, circ in circulos_final:
    cxi, cyi, ri = int(round(cx)), int(round(cy)), int(round(r))
    cv2.circle(mascara_interior, (cxi, cyi), ri, 255, -1)
    particulas.append({"id": len(particulas) + 1, "area_px": float(np.pi * ri * ri),
                       "circularidad": circ, "centro_x": cxi, "centro_y": cyi, "radio_eq_px": float(ri)})

# resultado final = TODAS las particulas en blanco (imagen binarizada lista para el watershed)
resultado_final = np.zeros((alto_px, ancho_px), dtype=np.uint8)
resultado_final[(mascara_interior > 0) | (mascara_exterior > 0)] = 255
print("Particulas interiores:", len(particulas), "(fusiones:", len(circulos) - len(particulas), ") | borde:", len(grupos_borde))

In [ ]:
# --- Guardar imagenes resultado ---
# 1) Resultado final binarizado (todas las particulas blancas, listo para el watershed)
cv2.imwrite(os.path.join(carpeta_salida, "resultado_final.png"), resultado_final)

# 2) Doble panel: interiores | exteriores (binario)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(mascara_interior, cmap="gray"); ax[0].set_title("Interiores: %d" % len(particulas)); ax[0].axis("off")
ax[1].imshow(mascara_exterior, cmap="gray"); ax[1].set_title("Exteriores / borde: %d" % len(grupos_borde)); ax[1].axis("off")
fig.tight_layout()
fig.savefig(os.path.join(carpeta_salida, "interiores_exteriores.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

# 3) Comparacion: resultado final vs la entrada del watershed (_WS)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(resultado_final, cmap="gray"); ax[0].set_title("Resultado final (binarizado)"); ax[0].axis("off")
ax[1].imshow(ws, cmap="gray"); ax[1].set_title("Entrada watershed (_WS)"); ax[1].axis("off")
fig.tight_layout()
fig.savefig(os.path.join(carpeta_salida, "comparacion.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

# 4) Overlay: resultado final superpuesto sobre el _WS (colores aptos deuteranopia)
#    blanco = coincide | naranja = anadido por el codigo | azul = quitado por el codigo
proc = resultado_final > 0
ws_fg = ws > 0
overlay_ws = np.zeros((alto_px, ancho_px, 3), dtype=np.uint8)
overlay_ws[ws_fg & proc] = (255, 255, 255)
overlay_ws[(~ws_fg) & proc] = (0, 128, 255)     # naranja (BGR) = anadido
overlay_ws[ws_fg & (~proc)] = (255, 0, 0)       # azul (BGR) = quitado
cv2.imwrite(os.path.join(carpeta_salida, "overlay_vs_ws.png"), overlay_ws)

# 5) Deteccion de particulas: cada particula dibujada por SEPARADO sobre la micrografia
#    original (si se fusionan en la mascara, findContours las une; por eso una a una).
#    interiores = circulo naranja | exteriores/borde = casco convexo azul (deuteranopia)
original = cv2.imread(ruta_original, cv2.IMREAD_GRAYSCALE)
deteccion = cv2.cvtColor(original, cv2.COLOR_GRAY2BGR)
for miembros in grupos_borde.values():
    cv2.drawContours(deteccion, [cv2.convexHull(np.vstack([contorno[p] for p in miembros]))], -1, (255, 0, 0), 2)
for p in particulas:
    cv2.circle(deteccion, (p["centro_x"], p["centro_y"]), int(round(p["radio_eq_px"])), (0, 128, 255), 2)
cv2.imwrite(os.path.join(carpeta_salida, "Deteccion de particulas.png"), deteccion)

print("Guardado en:", carpeta_salida)

In [ ]:
# --- Guardar las propiedades en Excel (solo particulas interiores) ---
libro = Workbook()
hoja = libro.active
hoja.title = "Resultados"

hoja.append(["RESUMEN"])
hoja.append(["Particulas interiores", len(particulas)])
hoja.append(["Particulas de borde (2o plano)", len(grupos_borde)])
hoja.append(["Fraccion de superficie", float((resultado_final > 0).mean())])
hoja.append([])
hoja.append(["id", "area_px", "area_um2", "diametro_eq_um", "circularidad", "centro_x_px", "centro_y_px"])
for p in particulas:
    hoja.append([
        p["id"], p["area_px"], p["area_px"] * micras_por_pixel_x * micras_por_pixel_y,
        2 * p["radio_eq_px"] * micras_por_pixel, p["circularidad"], p["centro_x"], p["centro_y"],
    ])
libro.save(os.path.join(carpeta_salida, "propiedades_particulas.xlsx"))
print("Excel guardado")